# Unified Perception — OWL-SAM Adapter Training (Phase C)

Trains the `OwlToSamAdapter` using **real OWL-ViT and SAM encoders**.

- OWL-ViT-B → patch features `(B, 576, 768)`
- SAM ViT-B → image features `(B, 256, 64, 64)` as training target
- Both encoders frozen; only the ~526K-param adapter trains
- Checkpoints saved every 100 batches → a Colab kernel restart never wipes progress

**GPU:** Enable T4 via `Runtime → Change runtime type → T4 GPU`

## 1. Check GPU

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Clone Repo

In [ ]:
!git clone https://github.com/sutharsan-311/unified-perception.git
%cd unified-perception
!ls

## 3. Install Dependencies

SAM is installed from the official Meta repo (the PyPI `segment-anything` package does **not** expose `sam_model_registry`).

In [ ]:
!pip install -q torchvision Pillow transformers
!pip install -q git+https://github.com/facebookresearch/segment-anything.git

## 4. Download COCO val2017 (~780 MB)

In [ ]:
import os
os.makedirs('data/coco', exist_ok=True)
!wget -q --show-progress -O data/coco/val2017.zip http://images.cocodataset.org/zips/val2017.zip
!unzip -q data/coco/val2017.zip -d data/coco/
!rm data/coco/val2017.zip
!ln -s val2017 data/coco/train2017 2>/dev/null || true
from pathlib import Path
print(f'Images: {len(list(Path("data/coco/val2017").glob("*.jpg")))}')


## 5. Download SAM ViT-B Checkpoint (~375 MB)

In [ ]:
!wget -q --show-progress -O sam_vit_b.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
print('SAM checkpoint ready.')

## 6. Verify Adapter Shape

In [ ]:
import sys
sys.path.insert(0, '.')
from nanosam.adapter import OwlToSamAdapter
device = 'cuda' if torch.cuda.is_available() else 'cpu'
adapter = OwlToSamAdapter().to(device)
print(f'Parameters: {adapter.get_parameter_count():,}')
x = torch.randn(2, 576, 768).to(device)
out = adapter(x)
assert out.shape == (2, 256, 64, 64)
print(f'Shape: {x.shape} -> {out.shape}  OK')

## 7. Train — Phase C (Real Encoders)

- OWL-ViT-B downloads from HuggingFace (~600 MB, first run only)
- `-u` streams logs live; checkpoints save every 100 batches
- If you hit a CUDA OOM, drop `--batch-size` to 1
- ~30-40 min on T4

In [ ]:
!python -u train.py \
  --dataset-dir data/coco \
  --num-images 1000 \
  --validation-size 100 \
  --batch-size 2 \
  --num-epochs 5 \
  --learning-rate 1e-3 \
  --device cuda \
  --checkpoint-dir checkpoints \
  --num-workers 0 \
  --log-interval 10 \
  --phase-c \
  --sam-checkpoint sam_vit_b.pth \
  --owl-model google/owlvit-base-patch32

## 7b. Resume After a Crash (optional)

If the kernel restarted mid-run, re-run cells 2–6, then run this to resume from the last periodic checkpoint instead of cell 7.

In [ ]:
import glob, os
ckpts = glob.glob('checkpoints/*_epoch*.pth')
resume = max(ckpts, key=os.path.getmtime) if ckpts else None
print('Resuming from:', resume)
if resume:
    !python -u train.py \
      --dataset-dir data/coco --num-images 1000 --validation-size 100 \
      --batch-size 2 --num-epochs 5 --learning-rate 1e-3 --device cuda \
      --checkpoint-dir checkpoints --num-workers 0 --log-interval 10 \
      --phase-c --sam-checkpoint sam_vit_b.pth \
      --owl-model google/owlvit-base-patch32 \
      --load-checkpoint "$resume"

## 8. Check Results

In [ ]:
import glob
checkpoints = sorted(glob.glob('checkpoints/*.pth'))
print('Saved checkpoints:')
for ck in checkpoints:
    print(f'  {ck}')
for name in ['checkpoints/adapter_phase_c_best.pth', 'checkpoints/adapter_phase_c_final.pth']:
    if os.path.exists(name):
        ck = torch.load(name, map_location=device, weights_only=False)
        print(f'\n{name}:')
        print(f'  Epoch:         {ck["epoch"]}')
        print(f'  Train losses:  {[round(l,4) for l in ck["train_losses"]]}')
        print(f'  Val losses:    {[round(l,4) for l in ck["val_losses"]]}')
        if ck['val_losses']:
            print(f'  Best val loss: {min(ck["val_losses"]):.6f}')

## 9. Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r checkpoints/ /content/drive/MyDrive/unified-perception-checkpoints/
print('Saved to Google Drive!')